In [1]:
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough

model="llama3.1"
llm = ChatOllama(model=model, temperature=0)

In [2]:
def addM(first:int, second:int):
    print(first+second)

    
    

In [3]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

store = {}

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a highly capable AI assistant tasked with understanding user queries and responding with the appropriate function from the list provided below. Your response must adhere to the following constraints:

            Functions:
                - addM(firstArgument, secondArgument)

            Constraints:
                - Only use the functions listed above. Do not generate or suggest any other functions.
                - Ensure that the function you generate directly addresses the query made by the user.
                - If the query does not correspond to any function you are allowed to use, respond with an empty string ''.
                - Include only the function name and arguments in your response, without any additional text.

            Examples:
                - Query: "Can you add 5 and 4?" 
                Response: "addM(5,4)"
                - Query: "What is the sum of 10 and 20?"
                Response: "addM(10,20)"
                - Query: "Subtract 30 from 100."
                Response: ''

            Be mindful that only the functions defined above are valid, and the response must match the function signature exactly.
            """
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain =  RunnablePassthrough.assign(messages=itemgetter("messages")) | prompt | llm 

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

config = {"configurable": {"session_id": "new"}}


In [15]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="can you subtract 200 and 550")]},
    config=config,
)

In [16]:
res = response.content.strip()
print(res)
if(len(res)>2):
    try:
        result = eval(res)
        print(f"Result: {result}")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Unexpected response:", res)

''
Unexpected response: ''
